<a href="https://colab.research.google.com/github/mooch443/dataset-fixer/blob/main/notebooks/03_fixed_cohort_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Fixed-cohort, cached model comparison

This tutorial demonstrates `Dataset.compare_models()`: two checkpoints are evaluated on exactly the same frozen validation images, annotations, class schema, and denominators. Predictions are cached independently from metrics and figures.

> **AI-generation disclosure:** this project and tutorial are largely AI-generated under human direction and review. Independently validate metrics and statistical assumptions for your work.

The example uses a pinned subset of official images and YOLO labels from the public [SAWIT dataset](https://github.com/dtnguyen0304/sawit), which declares the [MIT License](https://github.com/dtnguyen0304/sawit/blob/main/LICENSE). Ultralytics is installed as an optional inference/training dependency and retains its own upstream license. A GPU Colab runtime is recommended.

In [ ]:
import importlib, os, subprocess, sys, tempfile
from pathlib import Path

repo = next((candidate for candidate in [Path('/content/dataset-fixer'), Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / 'pyproject.toml').is_file()), None)
if repo is None:
    repo = Path(tempfile.mkdtemp(prefix='dataset-fixer-source-')) / 'dataset-fixer'
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/mooch443/dataset-fixer.git', str(repo)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{repo}[comparison]'])
sys.path.insert(0, str(repo / 'src'))
sys.path.insert(0, str(repo))
importlib.invalidate_caches()
import dataset_fixer
print('dataset-fixer:', dataset_fixer.__version__, dataset_fixer.__file__)
WORK_ROOT = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir()) / 'dataset-fixer-notebooks'
WORK_ROOT.mkdir(parents=True, exist_ok=True)

## 1. Create the immutable evaluation dataset

The downloader selects official SAWIT files deterministically from a pinned commit and creates fixed train/validation views without changing pixels or annotations. `Dataset.open()` validates dimensions, label schemas, coordinates, classes, and split consistency before either model is trained.

In [ ]:
from dataset_fixer import Dataset
from examples.download_public_examples import download_sawit_examples

example_root = WORK_ROOT / 'dataset-fixer-public-examples'
paths = download_sawit_examples(example_root, images_per_class=8)
evaluation_dataset = Dataset.open(paths['fixed'], task='detect', deep=True)
print(evaluation_dataset)
print((example_root / 'SOURCE.json').read_text()[:800])
evaluation_dataset.visualize(split='val', n=6, seed=42, columns=3)

## 2. Train two small comparison checkpoints

These short runs are intentionally different, not intended as strong models. The comparison engine treats each checkpoint plus its resolution and inference settings as one factual configuration.

In [ ]:
from ultralytics import YOLO

runs = WORK_ROOT / 'dataset-fixer-runs'
pretrained = WORK_ROOT / 'yolo11n.pt'
baseline_run = YOLO(str(pretrained)).train(
    data=str(evaluation_dataset.data_yaml), epochs=1, imgsz=320, batch=4,
    project=str(runs), name='baseline', seed=42, deterministic=True, exist_ok=True, verbose=False,
)
candidate_run = YOLO(str(pretrained)).train(
    data=str(evaluation_dataset.data_yaml), epochs=3, imgsz=320, batch=4,
    project=str(runs), name='candidate', seed=43, deterministic=True, exist_ok=True, verbose=False,
)
baseline_path = runs / 'baseline' / 'weights' / 'best.pt'
candidate_path = runs / 'candidate' / 'weights' / 'best.pt'
assert baseline_path.is_file() and candidate_path.is_file()

## 3. Compare on one frozen validation cohort

The cohort is resolved once from `evaluation_dataset`. The engine fails on missing, added, skipped, or reordered images. Training provenance is checked against ultimate originals, while threshold selection is explicitly labeled as validation/model selection.

In [ ]:
import shutil
comparison_destination = WORK_ROOT / 'sawit-model-comparison'
if comparison_destination.exists():
    shutil.rmtree(comparison_destination)

comparison = evaluation_dataset.compare_models(
    models={
        'baseline-1epoch': {
            'path': baseline_path,
            'training_dataset': evaluation_dataset.location,
            'resolution': 320,
        },
        'candidate-3epochs': {
            'path': candidate_path,
            'training_dataset': evaluation_dataset.location,
            'resolution': 320,
        },
    },
    split='val',
    baseline='baseline-1epoch',
    inference='native',
    protocol='validation',
    training_provenance='required',
    confidence_thresholds=(0.15, 0.25, 0.35, 0.50),
    postprocess_thresholds=(0.50, 0.70),
    destination=comparison_destination,
    cache=True,
    visualize=True,
)
print(comparison)
print('cohort fingerprint:', comparison.cohort_fingerprint)
print('cohort verified:', comparison.cohort_verified)
print('training overlap detected:', comparison.training_overlap_detected)
print('cache statistics:', comparison.cache_statistics)

## 4. Inspect exact tables and generated figures

Every figure is accompanied by its plotted CSV values and metadata JSON. The metadata includes the cohort fingerprint, model hashes, backend, cache source, independent-cluster count, and metric definition.

In [ ]:
import pandas as pd
from IPython.display import Image as DisplayImage, display

ranking = pd.read_csv(comparison.location / 'metrics' / 'ranking.csv')
paired = pd.read_csv(comparison.location / 'metrics' / 'paired_statistics.csv')
display(ranking)
display(paired)
display(DisplayImage(filename=str(comparison.location / 'figures' / 'ranking_forest.png'), width=900))
display(DisplayImage(filename=str(comparison.location / 'qualitative' / 'comparison_01.png'), width=1200))

### Reusing the prediction cache

Run the comparison again with a new output destination and identical model/cohort/inference settings. Prediction shards are reused, while metrics and figures are regenerated. Changing model bytes, image or annotation content, backend versions, resolution, slicing, overlap, precision, augmentation, or postprocessing invalidates the relevant cache key.

For final held-out reporting, use `protocol='calibrate_then_test'` with distinct calibration and test splits, or `protocol='locked'` with predetermined settings.